# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Notebook 7 – Semantic Enrichment Using Maqāyīs al-Lughah (DD-008)

## Introduction

The corrected Point-2 pipeline established the authoritative query-root representation by accepting valid Quranic Arabic Corpus (QAC) roots first and using strictly validated CAMeL roots only as fallback. Each resolved query term retains its selected root, root source, resolution status, and supporting QAC/CAMeL provenance; unresolved terms remain unresolved.

This notebook does not generate, infer, repair, or replace query roots. It attaches lexical-semantic evidence from *Maqāyīs al-Lughah* to the authoritative roots produced by Point 2. Classical Arabic lexicons, particularly *Maqāyīs al-Lughah* by Ibn Fāris, organize vocabulary around semantic root principles rather than isolated word definitions. Each indexed root is associated with one or more fundamental semantic concepts from which related lexical meanings are derived.

The resulting semantically enriched dataset preserves the complete Point-2 query-root lineage and serves as the query-side semantic representation for later retrieval stages. A Maqāyīs match is lexical-semantic evidence attached to an existing authoritative root; it is not a new root decision, a retrieval model, or proof of passage relevance.

---

## Objectives

The objectives of this notebook are to:

1. Load the final corrected Point-2 hybrid-root artifact.
2. Preserve the authoritative query term, root, source, resolution status, and QAC/CAMeL provenance.
3. Connect to the existing *Maqāyīs al-Lughah* lexical database.
4. Build a deterministic exact-root lookup structure.
5. Retrieve semantic evidence only for query terms already marked root-resolved by Point 2.
6. Attach semantic evidence without changing or reconstructing any query root.
7. Report semantic coverage by root source and overall.
8. Export an isolated corrected semantic-enrichment dataset for later validation and downstream use.

---

## Expected Deliverables

At the completion of this notebook, the following deliverables are produced:

- A deterministic semantic lookup structure derived from *Maqāyīs al-Lughah*.
- Semantic evidence attached only to authoritative Point-2 roots.
- A semantically enriched QuranQA query representation that preserves Point-2 provenance.
- Overall and QAC/CAMeL-specific coverage diagnostics.
- An isolated corrected semantic-enrichment CSV.

---

## Design Decision (DD-008)

The framework uses *Maqāyīs al-Lughah* as a root-oriented lexical-semantic evidence source. Lookup is performed with the unchanged authoritative Point-2 root. Missing lexicon entries remain missing, and unresolved query terms are not assigned or guessed roots during this stage.

In [1]:
from pathlib import Path
import sys


def discover_project_root(start: Path) -> Path:
    """Locate the workspace from its authority marker and expected layout."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        marker = candidate / "00_Project_Control" / "README_Workspace_Authority.txt"
        point2 = (
            candidate
            / "02_Methodology_Evidence"
            / "QAC and Root Extraction"
            / "Corrected_Outputs"
            / "Point_2_Query_Root_Provenance"
            / "QuranQA_Hybrid_Root_Retrieval_Corrected.csv"
        )
        if marker.exists() and point2.exists():
            return candidate
    raise FileNotFoundError("Could not locate the thesis workspace from the current notebook directory")


NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = discover_project_root(NOTEBOOK_DIR)
SEMANTIC_CODE_DIR = PROJECT_ROOT / "02_Methodology_Evidence" / "Maqāyīs Lookup"

if str(SEMANTIC_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SEMANTIC_CODE_DIR))

import semantic_lookup as sl

print("=" * 70)
print("SEMANTIC LOOKUP MODULE")
print("=" * 70)
print(f"Project Root    : {PROJECT_ROOT}")
print(f"Semantic Helper : {Path(sl.__file__).resolve()}")

SEMANTIC LOOKUP MODULE
Project Root    : ${THESIS_WORKSPACE}
Semantic Helper : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Maqāyīs Lookup\semantic_lookup.py


In [2]:
# ============================================================
# Import Required Libraries
# ============================================================

from pathlib import Path
import sqlite3
import json

import numpy as np
import pandas as pd

from IPython.display import display

# ------------------------------------------------------------
# Display Options
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("=" * 70)
print("LIBRARIES LOADED")
print("=" * 70)

print("✓ pathlib")
print("✓ sqlite3")
print("✓ json")
print("✓ numpy")
print("✓ pandas")
print("✓ IPython.display")

LIBRARIES LOADED
✓ pathlib
✓ sqlite3
✓ json
✓ numpy
✓ pandas
✓ IPython.display


In [3]:
# ============================================================
# Define Current Project Paths
# ============================================================

METHODOLOGY_DIR = PROJECT_ROOT / "02_Methodology_Evidence"
ROOT_EXTRACTION_DIR = METHODOLOGY_DIR / "QAC and Root Extraction"
LEXICON_DIR = METHODOLOGY_DIR / "Maqāyīs Lookup" / "Lexicon Database and Metadata"
OUTPUT_DIR = (
    METHODOLOGY_DIR
    / "Maqāyīs Lookup"
    / "Corrected_Outputs"
    / "Notebook_7_Maqayis_Semantic_Enrichment"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HYBRID_ROOT_FILE = (
    ROOT_EXTRACTION_DIR
    / "Corrected_Outputs"
    / "Point_2_Query_Root_Provenance"
    / "QuranQA_Hybrid_Root_Retrieval_Corrected.csv"
)
MAQAYIS_DB_FILE = LEXICON_DIR / "db.sqlite"
OUTPUT_FILE = OUTPUT_DIR / "QuranQA_Semantic_Enrichment_Corrected.csv"

if not HYBRID_ROOT_FILE.exists():
    raise FileNotFoundError(HYBRID_ROOT_FILE)
if not MAQAYIS_DB_FILE.exists():
    raise FileNotFoundError(MAQAYIS_DB_FILE)

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)
print(f"Authoritative Point-2 Input : {HYBRID_ROOT_FILE}")
print(f"Maqāyīs Database           : {MAQAYIS_DB_FILE}")
print(f"Corrected Output           : {OUTPUT_FILE}")

PROJECT PATHS
Authoritative Point-2 Input : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance\QuranQA_Hybrid_Root_Retrieval_Corrected.csv
Maqāyīs Database           : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Maqāyīs Lookup\Lexicon Database and Metadata\db.sqlite
Corrected Output           : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Maqāyīs Lookup\Corrected_Outputs\Notebook_7_Maqayis_Semantic_Enrichment\QuranQA_Semantic_Enrichment_Corrected.csv


In [4]:
# ============================================================
# Load Hybrid Root Retrieval Results
# ============================================================

hybrid_df = pd.read_csv(HYBRID_ROOT_FILE)

print("=" * 70)
print("HYBRID ROOT RETRIEVAL RESULTS LOADED")
print("=" * 70)

print(f"Total Questions : {len(hybrid_df):,}")
print(f"Total Columns   : {len(hybrid_df.columns)}")

print("\nColumns")

print(list(hybrid_df.columns))

print("\nSample Record")

display(hybrid_df.head(1))

HYBRID ROOT RETRIEVAL RESULTS LOADED
Total Questions : 251
Total Columns   : 6

Columns
['Dataset', 'Question_ID', 'Question', 'Query_Terms', 'Query_Term_Provenance', 'Lexical_Matches']

Sample Record


,Dataset,Question_ID,Question,Query_Terms,Query_Term_Provenance,Lexical_Matches
0,Train,101,من هم قوم شعيب؟,"[""قوم"", ""شعيب""]","[{""Token_Position"": 2, ""Original_Term"": ""قوم"", ""Normalized_Term"": ""قوم""}, {""Token_Position"": 3, ""Original_Term"": ""شع...","[{""Query_Term"": ""قوم"", ""Original_Term"": ""قوم"", ""Normalized_Term"": ""قوم"", ""Matched"": true, ""Root_Resolved"": true, ""Re..."


## Parsing the Authoritative Point-2 Representation

The final corrected Point-2 CSV stores `Query_Terms`, `Query_Term_Provenance`, and `Lexical_Matches` as standards-compliant JSON. These structures are parsed without modifying their values.

`Root_Resolved`, `Root_AR`, `Root_Source`, and `Resolution_Status` remain authoritative. Notebook 7 does not reinterpret legacy `Matched` flags, rerun morphology, or independently validate and replace roots. Only records already marked root-resolved by Point 2 are eligible for Maqāyīs lookup.

In [5]:
# ============================================================
# Parse and Validate Authoritative Point-2 Structures
# ============================================================

NESTED_COLUMNS = ["Query_Terms", "Query_Term_Provenance", "Lexical_Matches"]

for column in NESTED_COLUMNS:
    hybrid_df[column] = hybrid_df[column].apply(
        lambda value: json.loads(value) if isinstance(value, str) else value
    )
    if not hybrid_df[column].map(lambda value: isinstance(value, list)).all():
        raise TypeError(f"{column} must contain one list per question")

term_records = []
for _, row in hybrid_df.iterrows():
    matches = row["Lexical_Matches"]
    provenance = row["Query_Term_Provenance"]
    if len(matches) != len(row["Query_Terms"]) or len(matches) != len(provenance):
        raise ValueError(f"Nested query-term lengths disagree for Question_ID={row['Question_ID']}")
    for position, (match, term_provenance) in enumerate(zip(matches, provenance), start=1):
        term_records.append({
            "Dataset": row["Dataset"],
            "Question_ID": row["Question_ID"],
            "Question": row["Question"],
            "Term_Order": position,
            "Token_Position": term_provenance.get("Token_Position"),
            "Original_Term": match.get("Original_Term", term_provenance.get("Original_Term")),
            "Normalized_Term": match.get("Normalized_Term", term_provenance.get("Normalized_Term")),
            "Root_Resolved": bool(match.get("Root_Resolved", False)),
            "Resolution_Status": match.get("Resolution_Status"),
            "Root_AR": match.get("Root_AR"),
            "Root_Source": match.get("Root_Source"),
        })

term_lineage_df = pd.DataFrame(term_records)
resolved_mask = term_lineage_df["Root_Resolved"]

print("=" * 70)
print("AUTHORITATIVE POINT-2 REPRESENTATION PARSED")
print("=" * 70)
print(f"Questions             : {len(hybrid_df):,}")
print(f"Total Query Terms     : {len(term_lineage_df):,}")
print(f"Strict Resolved Roots : {int(resolved_mask.sum()):,}")
print(f"Unresolved Terms      : {int((~resolved_mask).sum()):,}")
print("\nResolved Roots by Source")
display(term_lineage_df.loc[resolved_mask, "Root_Source"].value_counts().rename_axis("Root_Source").reset_index(name="Terms"))

AUTHORITATIVE POINT-2 REPRESENTATION PARSED
Questions             : 251
Total Query Terms     : 1,342
Strict Resolved Roots : 1,013
Unresolved Terms      : 329

Resolved Roots by Source


,Root_Source,Terms
0,QAC,525
1,CAMeL,488


## Extracting the Authoritative Root Inventory

Maqāyīs lookup is performed only for query terms whose Point-2 record has `Root_Resolved=True`. The selected `Root_AR` and `Root_Source` are copied unchanged into a trace table. Unresolved terms are excluded from lookup without being assigned a replacement root.

Repeated roots are consolidated only to avoid redundant lexicon lookups. This deduplication does not select, normalize, infer, or replace a root; every occurrence retains its original Point-2 provenance in the final nested representation.

In [6]:
# ============================================================
# Extract the Authoritative Root Inventory
# ============================================================

root_records = []

for _, row in hybrid_df.iterrows():
    for position, (match, term_provenance) in enumerate(
        zip(row["Lexical_Matches"], row["Query_Term_Provenance"]), start=1
    ):
        resolved = bool(match.get("Root_Resolved", False))
        root = match.get("Root_AR")
        source = match.get("Root_Source")

        if not resolved:
            if root is not None and str(root).strip():
                raise ValueError("An unresolved Point-2 term contains a nonblank authoritative root")
            continue

        if root is None or not str(root).strip():
            raise ValueError("A resolved Point-2 term has a blank authoritative root")
        if source not in {"QAC", "CAMeL"}:
            raise ValueError(f"A resolved Point-2 term has invalid Root_Source={source!r}")

        root_records.append({
            "Dataset": row["Dataset"],
            "Question_ID": row["Question_ID"],
            "Question": row["Question"],
            "Term_Order": position,
            "Token_Position": term_provenance.get("Token_Position"),
            "Original_Term": match.get("Original_Term", term_provenance.get("Original_Term")),
            "Normalized_Term": match.get("Normalized_Term", term_provenance.get("Normalized_Term")),
            "Root_AR": str(root),
            "Root_Source": source,
            "Resolution_Status": match.get("Resolution_Status"),
            "Lemma_AR": match.get("Lemma_AR"),
            "POS": match.get("POS"),
        })

roots_df = pd.DataFrame(root_records)

unique_roots_df = (
    roots_df.sort_values(
        ["Root_AR", "Dataset", "Question_ID", "Term_Order"],
        kind="mergesort",
    )
    .drop_duplicates(subset=["Root_AR"], keep="first")
    .reset_index(drop=True)
)

print("=" * 70)
print("AUTHORITATIVE ROOT INVENTORY EXTRACTED")
print("=" * 70)
print(f"Resolved Root Occurrences : {len(roots_df):,}")
print(f"Unique Authoritative Roots: {len(unique_roots_df):,}")
print("\nRoot Occurrences by Source")
display(roots_df["Root_Source"].value_counts().rename_axis("Root_Source").reset_index(name="Terms"))
print("\nSample Authoritative Roots")
display(unique_roots_df[["Root_AR", "Root_Source", "Original_Term", "Normalized_Term", "Lemma_AR", "POS"]].head(20))

AUTHORITATIVE ROOT INVENTORY EXTRACTED
Resolved Root Occurrences : 1,013
Unique Authoritative Roots: 393

Root Occurrences by Source


,Root_Source,Terms
0,QAC,525
1,CAMeL,488



Sample Authoritative Roots


,Root_AR,Root_Source,Original_Term,Normalized_Term,Lemma_AR,POS
0,ابو,QAC,ابو,ابو,أَبٌ,N
1,اثم,QAC,الإثم,الاثم,إِثْم,N
2,اخذ,QAC,اتخاذ,اتخاذ,ٱتِّخَاذ,N
3,اخو,QAC,اخو,اخو,أَخ,N
4,ارض,QAC,الأرض,الارض,أَرْض,N
5,افك,QAC,الإفك,الافك,إِفْك,N
6,اكل,QAC,يأكل,ياكل,أَكَلَ,V
7,الل,QAC,الا,الا,إِلّ,N
8,اله,QAC,الله,الله,ٱللَّه,PN
9,امر,QAC,الأمور,الامور,أَمْر,N


## Loading the *Maqāyīs al-Lughah* Lexicon

The semantic enrichment process requires efficient access to the lexical entries contained in *Maqāyīs al-Lughah*. Rather than executing repeated database queries for each extracted root, the complete lexicon is loaded into memory once at the beginning of the enrichment stage.

Since the lexicon contains a relatively small number of entries, loading the entire dataset into an in-memory data structure substantially improves lookup efficiency while simplifying the implementation. The loaded lexical entries will subsequently be transformed into a root-based lookup dictionary for semantic retrieval.

In [7]:
# ============================================================
# Load Maqāyīs al-Lughah Lexicon Deterministically
# ============================================================

connection = sqlite3.connect(MAQAYIS_DB_FILE)
maqayis_df = pd.read_sql_query(
    '''
    SELECT word, meanings
    FROM maqayeesul_luga
    ORDER BY id
    ''',
    connection,
)
connection.close()

print("=" * 70)
print("MAQĀYĪS AL-LUGHAH LOADED")
print("=" * 70)
print(f"Total Lexicon Entries : {len(maqayis_df):,}")
print(f"Unique Trimmed Roots  : {maqayis_df['word'].astype(str).str.strip().nunique():,}")
print(f"Blank Roots           : {int(maqayis_df['word'].isna().sum() + maqayis_df['word'].astype(str).str.strip().eq('').sum()):,}")
print(f"Blank Meanings        : {int(maqayis_df['meanings'].isna().sum() + maqayis_df['meanings'].astype(str).str.strip().eq('').sum()):,}")
print("\nSample Entries")
display(maqayis_df.head(10))

MAQĀYĪS AL-LUGHAH LOADED
Total Lexicon Entries : 5,274
Unique Trimmed Roots  : 5,274
Blank Roots           : 0
Blank Meanings        : 0

Sample Entries


,word,meanings
0,أب,(أَبَّ) اعْلَمْ أَنَّ لِلْهَمْزَةِ وَالْبَاءِ فِي الْمُضَاعَفِ أَصْلَيْنِ: أَحَدُهُمَا الْمَرْعَى، وَالْآخَرُ الْقَص...
1,أبت,(أَبَتَ) الْهَمْزَةُ وَالْبَاءُ وَالتَّاءُ أَصْلٌ وَاحِدٌ، وَهُوَ الْحَرُّ وَشِدَّتُهُ. قَالَ ابْنُ السِّكِّيتِ وَغَ...
2,أبث,(أَبَثَ) وَهَذَا الْبَابُ مُهْمَلٌ عِنْدَ الْخَلِيلِ. قَالَ الشَّيْبَانِيُّ: الْأَبِثُ: الْأَشِرُ النَّشِيطُ. قَالَ:...
3,أبد,(أَبَدَ) الْهَمْزَةُ وَالْبَاءُ وَالدَّالُ يَدُلُّ بِنَاؤُهَا عَلَى طُولِ الْمُدَّةِ، وَعَلَى التَّوَحُّشِ. قَالُوا:...
4,أبر,(أَبَرَ) الْهَمْزَةُ وَالْبَاءُ وَالرَّاءُ يَدُلُّ بِنَاؤُهَا عَلَى نَخْسِ الشَّيْءِ بِشَيْءٍ مُحَدَّدٍ. قَالَ الْخَ...
5,أبز,(أَبَزَ) الْهَمْزَةُ وَالْبَاءُ وَالزَّاءُ يَدُلُّ عَلَى الْقَلَقِ وَالسُّرْعَةِ وَقِلَّةِ الِاسْتِقْرَارِ. قَالَ ال...
6,أبس,(أَبَسَ) الْهَمْزَةُ وَالْبَاءُ وَالسِّينُ تَدُلُّ عَلَى الْقَهْرِ، يُقَالُ مِنْهُ أَبَسَ الرَّجُلُ الرَّجُلَ: إِذَا...
7,أبش,(أَبَشَ) الْهَمْزَةُ وَالْبَاءُ وَالشِّينُ لَيْسَ بِأَصْلٍ، لِأَنَّ الْهَمْزَةَ فِيهِ مُبْدَلَةٌ مِنْ هَاءٍ. قَالَ ا...
8,أبض,(أَبَضَ) الْهَمْزَةُ وَالْبَاءُ وَالضَّادُ تَدُلُّ عَلَى الدَّهْرِ، وَعَلَى شَيْءٍ مِنْ أَرْفَاغِ الْبَطْنِ. الْأُبْ...
9,أبط,(أَبَطَ) الْهَمْزَةُ وَالْبَاءُ وَالطَّاءُ أَصْلٌ وَاحِدٌ، وَهُوَ إِبِطُ الْإِنْسَانِ أَوِ اسْتِعَارَةٌ فِي غَيْرِهِ...


## Building the Semantic Root Lookup Dictionary

To support efficient semantic enrichment, the lexical entries loaded from *Maqāyīs al-Lughah* are transformed into an in-memory lookup dictionary indexed by Arabic roots.

Each dictionary entry maps a lexical root to its corresponding semantic explanation provided by Ibn Fāris. This structure enables constant-time retrieval of semantic information during the enrichment stage and avoids repeated searches through the lexicon.

Since every root is stored only once, the resulting lookup dictionary serves as the semantic index for the remainder of the notebook.

In [8]:
# ============================================================
# Build Maqāyīs Semantic Lookup Dictionary
# ============================================================
# ------------------------------------------------------------
# Build Lookup Dictionary
# ------------------------------------------------------------
maqayis_lookup, duplicate_entries = (
    sl.build_maqayis_lookup(maqayis_df)
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("=" * 70)
print("SEMANTIC LOOKUP DICTIONARY CREATED")
print("=" * 70)

print(f"Dictionary Entries : {len(maqayis_lookup):,}")
print(f"Duplicate Entries  : {duplicate_entries:,}")

# ------------------------------------------------------------
# Sample Lookup Entries
# ------------------------------------------------------------

sample_roots = list(maqayis_lookup.keys())[:10]

sample_df = pd.DataFrame({

    "Root": sample_roots,

    "Semantic_Meaning": [

        maqayis_lookup[root]

        for root in sample_roots

    ]

})

print("\nSample Dictionary Entries")

display(sample_df)

SEMANTIC LOOKUP DICTIONARY CREATED
Dictionary Entries : 5,274
Duplicate Entries  : 0

Sample Dictionary Entries


,Root,Semantic_Meaning
0,أب,(أَبَّ) اعْلَمْ أَنَّ لِلْهَمْزَةِ وَالْبَاءِ فِي الْمُضَاعَفِ أَصْلَيْنِ: أَحَدُهُمَا الْمَرْعَى، وَالْآخَرُ الْقَص...
1,أبت,(أَبَتَ) الْهَمْزَةُ وَالْبَاءُ وَالتَّاءُ أَصْلٌ وَاحِدٌ، وَهُوَ الْحَرُّ وَشِدَّتُهُ. قَالَ ابْنُ السِّكِّيتِ وَغَ...
2,أبث,(أَبَثَ) وَهَذَا الْبَابُ مُهْمَلٌ عِنْدَ الْخَلِيلِ. قَالَ الشَّيْبَانِيُّ: الْأَبِثُ: الْأَشِرُ النَّشِيطُ. قَالَ:...
3,أبد,(أَبَدَ) الْهَمْزَةُ وَالْبَاءُ وَالدَّالُ يَدُلُّ بِنَاؤُهَا عَلَى طُولِ الْمُدَّةِ، وَعَلَى التَّوَحُّشِ. قَالُوا:...
4,أبر,(أَبَرَ) الْهَمْزَةُ وَالْبَاءُ وَالرَّاءُ يَدُلُّ بِنَاؤُهَا عَلَى نَخْسِ الشَّيْءِ بِشَيْءٍ مُحَدَّدٍ. قَالَ الْخَ...
5,أبز,(أَبَزَ) الْهَمْزَةُ وَالْبَاءُ وَالزَّاءُ يَدُلُّ عَلَى الْقَلَقِ وَالسُّرْعَةِ وَقِلَّةِ الِاسْتِقْرَارِ. قَالَ ال...
6,أبس,(أَبَسَ) الْهَمْزَةُ وَالْبَاءُ وَالسِّينُ تَدُلُّ عَلَى الْقَهْرِ، يُقَالُ مِنْهُ أَبَسَ الرَّجُلُ الرَّجُلَ: إِذَا...
7,أبش,(أَبَشَ) الْهَمْزَةُ وَالْبَاءُ وَالشِّينُ لَيْسَ بِأَصْلٍ، لِأَنَّ الْهَمْزَةَ فِيهِ مُبْدَلَةٌ مِنْ هَاءٍ. قَالَ ا...
8,أبض,(أَبَضَ) الْهَمْزَةُ وَالْبَاءُ وَالضَّادُ تَدُلُّ عَلَى الدَّهْرِ، وَعَلَى شَيْءٍ مِنْ أَرْفَاغِ الْبَطْنِ. الْأُبْ...
9,أبط,(أَبَطَ) الْهَمْزَةُ وَالْبَاءُ وَالطَّاءُ أَصْلٌ وَاحِدٌ، وَهُوَ إِبِطُ الْإِنْسَانِ أَوِ اسْتِعَارَةٌ فِي غَيْرِهِ...


## Building the Semantic Root Table

The extracted Arabic roots are semantically enriched by retrieving their corresponding entries from *Maqāyīs al-Lughah*. Each root is matched against the semantic lookup dictionary constructed in the previous step.

The resulting semantic root table provides a one-to-one mapping between each validated Arabic root and its semantic explanation. In addition to the retrieved meaning, the table records whether a semantic entry was successfully found in the lexicon.

This intermediate resource enables the semantic enrichment process to be evaluated independently before the retrieved meanings are integrated into the complete query representation.

In [9]:
# ============================================================
# Build Semantic Root Table Using Exact Authoritative Roots
# ============================================================

semantic_roots_df = sl.lookup_semantic_meanings(unique_roots_df, maqayis_lookup)
semantic_roots_df["Maqayis_Lookup_Attempted"] = True
semantic_roots_df["Maqayis_Lookup_Root"] = semantic_roots_df["Root_AR"]

found_unique = int(semantic_roots_df["Semantic_Found"].sum())
missing_unique = len(semantic_roots_df) - found_unique
unique_coverage = 100 * found_unique / len(semantic_roots_df) if len(semantic_roots_df) else 0.0

root_occurrence_semantics_df = roots_df.merge(
    semantic_roots_df[["Root_AR", "Semantic_Found", "Semantic_Meaning"]],
    on="Root_AR",
    how="left",
    validate="many_to_one",
)

print("=" * 70)
print("SEMANTIC ROOT TABLE CREATED")
print("=" * 70)
print(f"Unique Roots            : {len(semantic_roots_df):,}")
print(f"Unique Semantic Matches : {found_unique:,}")
print(f"Unique Roots Without Match: {missing_unique:,}")
print(f"Unique-Root Coverage     : {unique_coverage:.2f}%")
print("\nSample Semantic Entries")
display(semantic_roots_df.head(20))

SEMANTIC ROOT TABLE CREATED
Unique Roots            : 393
Unique Semantic Matches : 295
Unique Roots Without Match: 98
Unique-Root Coverage     : 75.06%

Sample Semantic Entries


,Root_AR,Root_Source,Lemma_AR,POS,Semantic_Found,Semantic_Meaning,Maqayis_Lookup_Attempted,Maqayis_Lookup_Root
0,ابو,QAC,أَبٌ,N,False,NaN,True,ابو
1,اثم,QAC,إِثْم,N,False,NaN,True,اثم
2,اخذ,QAC,ٱتِّخَاذ,N,False,NaN,True,اخذ
3,اخو,QAC,أَخ,N,False,NaN,True,اخو
4,ارض,QAC,أَرْض,N,False,NaN,True,ارض
5,افك,QAC,إِفْك,N,False,NaN,True,افك
6,اكل,QAC,أَكَلَ,V,False,NaN,True,اكل
7,الل,QAC,إِلّ,N,False,NaN,True,الل
8,اله,QAC,ٱللَّه,PN,False,NaN,True,اله
9,امر,QAC,أَمْر,N,False,NaN,True,امر


## Analyzing Roots Without Semantic Matches

Although the majority of the extracted Arabic roots were successfully matched with entries in *Maqāyīs al-Lughah*, a subset of roots remained unmatched.

This section investigates the unmatched roots to better understand the limitations of the semantic enrichment process. The analysis identifies the roots that could not be linked to the lexicon, summarizes their frequency, and examines their lexical characteristics, including their source, lemma, and part of speech.

The findings provide insight into whether the missing semantic entries are caused by lexical gaps in the dictionary, differences in orthographic representation, or artifacts introduced during morphological analysis.

In [10]:
# ============================================================
# Analyze Unmatched Semantic Roots
# ============================================================

missing_semantics_df = semantic_roots_df.loc[~semantic_roots_df["Semantic_Found"]].copy()
missing_occurrences_df = root_occurrence_semantics_df.loc[
    ~root_occurrence_semantics_df["Semantic_Found"]
].copy()

source_summary = (
    missing_occurrences_df["Root_Source"]
    .value_counts()
    .rename_axis("Root_Source")
    .reset_index(name="Unmatched_Terms")
)
pos_summary = (
    missing_occurrences_df["POS"]
    .value_counts(dropna=False)
    .rename_axis("POS")
    .reset_index(name="Frequency")
)

print("=" * 70)
print("UNMATCHED SEMANTIC ROOTS")
print("=" * 70)
print(f"Unique Roots Without Match : {len(missing_semantics_df):,}")
print(f"Term Occurrences Without Match: {len(missing_occurrences_df):,}")
print("\nUnmatched Term Occurrences by Root Source")
display(source_summary)
print("\nUnmatched Term Occurrences by Part of Speech")
display(pos_summary)
print("\nDetailed Unique Roots Without Match")
display(missing_semantics_df[["Root_AR", "Root_Source", "Lemma_AR", "POS"]].sort_values("Root_AR"))

UNMATCHED SEMANTIC ROOTS
Unique Roots Without Match : 98
Term Occurrences Without Match: 289

Unmatched Term Occurrences by Root Source


,Root_Source,Unmatched_Terms
0,QAC,152
1,CAMeL,137



Unmatched Term Occurrences by Part of Speech


,POS,Frequency
0,noun,75
1,N,72
2,PN,47
3,V,26
4,verb,25
5,noun_prop,10
6,ADJ,7
7,prep,6
8,noun_quant,6
9,adv,5



Detailed Unique Roots Without Match


,Root_AR,Root_Source,Lemma_AR,POS
0,ابو,QAC,أَبٌ,N
1,اثم,QAC,إِثْم,N
2,اخذ,QAC,ٱتِّخَاذ,N
3,اخو,QAC,أَخ,N
4,ارض,QAC,أَرْض,N
...,...,...,...,...
371,نهة,CAMeL,نِهايَة,adv
379,ه,CAMeL,هُوَ,pron
380,هدهد,QAC,هُدْهُد,N
382,هلل,CAMeL,أَهَلّ,verb


## Diagnostic Analysis of Unmatched Roots

The semantic lookup relies on exact equality between the unchanged authoritative Point-2 root and the trimmed `word` field in *Maqāyīs al-Lughah*. Differences in orthographic representation or lexical indexing may therefore leave a valid root without a semantic match.

The partial-string search below is diagnostic evidence only. Its candidates do not alter the authoritative root, do not count as semantic matches, and are not used to enrich unresolved terms.

In [11]:
# ============================================================
# Diagnostic Search for Unmatched Roots (Non-Enriching)
# ============================================================

connection = sqlite3.connect(MAQAYIS_DB_FILE)
roots_to_check = missing_semantics_df["Root_AR"].sort_values(kind="mergesort").tolist()
diagnostic_results = []

for root in roots_to_check:
    matches = pd.read_sql_query(
        '''
        SELECT word
        FROM maqayeesul_luga
        WHERE word LIKE ?
        ORDER BY word
        LIMIT 10
        ''',
        connection,
        params=[f"%{root}%"],
    )
    diagnostic_results.append({
        "Root_AR": root,
        "Found_Diagnostic_Candidate": len(matches) > 0,
        "Candidate_Entries": ", ".join(matches["word"].astype(str).tolist()),
    })

connection.close()
diagnostic_df = pd.DataFrame(diagnostic_results)

print("=" * 70)
print("NON-ENRICHING DIAGNOSTIC SEARCH RESULTS")
print("=" * 70)
print(f"Roots Investigated : {len(diagnostic_df):,}")
print(f"Potential Candidates: {int(diagnostic_df['Found_Diagnostic_Candidate'].sum()):,}")
print(f"No Candidates      : {int((~diagnostic_df['Found_Diagnostic_Candidate']).sum()):,}")
display(diagnostic_df.head(30))

NON-ENRICHING DIAGNOSTIC SEARCH RESULTS
Roots Investigated : 98
Potential Candidates: 16
No Candidates      : 82


,Root_AR,Found_Diagnostic_Candidate,Candidate_Entries
0,ابو,False,
1,اثم,False,
2,اخذ,False,
3,اخو,False,
4,ارض,False,
5,افك,False,
6,اكل,False,
7,الل,True,"اللهجم, اللهذم"
8,اله,True,"الهبرقي, الهبركة, الهبلع, الهبنقع, الهجرس, الهجرع, الهجنع, الهدلق, الهدملة, الهذرمة"
9,امر,False,


## Attaching Maqāyīs Evidence Without Changing Query Roots

The exact-root semantic table is attached to a copy of every Point-2 lexical record. All original query-term and root provenance fields remain unchanged.

Only records with `Root_Resolved=True` receive a Maqāyīs lookup attempt. The lookup key is recorded explicitly and must equal the authoritative `Root_AR`. Unresolved terms retain `Maqayis_Lookup_Attempted=False` and receive no guessed root or semantic evidence.

In [12]:
# ============================================================
# Attach Semantic Evidence While Preserving Point-2 Lineage
# ============================================================

meaning_by_root = dict(zip(semantic_roots_df["Root_AR"], semantic_roots_df["Semantic_Meaning"]))
enriched_records = []

for _, row in hybrid_df.iterrows():
    enriched_matches = []
    for match in row["Lexical_Matches"]:
        enriched_match = match.copy()
        resolved = bool(match.get("Root_Resolved", False))
        root = match.get("Root_AR")

        if resolved:
            meaning = meaning_by_root.get(root)
            semantic_found = bool(pd.notna(meaning))
            if not semantic_found:
                meaning = None
            enriched_match["Maqayis_Lookup_Attempted"] = True
            enriched_match["Maqayis_Lookup_Root"] = root
            enriched_match["Semantic_Found"] = semantic_found
            enriched_match["Semantic_Meaning"] = meaning
            enriched_match["Semantic_Source"] = "Maqāyīs al-Lughah" if semantic_found else None
        else:
            enriched_match["Maqayis_Lookup_Attempted"] = False
            enriched_match["Maqayis_Lookup_Root"] = None
            enriched_match["Semantic_Found"] = False
            enriched_match["Semantic_Meaning"] = None
            enriched_match["Semantic_Source"] = None

        enriched_matches.append(enriched_match)

    enriched_row = row.to_dict()
    enriched_row["Lexical_Matches"] = enriched_matches
    enriched_records.append(enriched_row)

semantic_enrichment_df = pd.DataFrame(enriched_records, columns=hybrid_df.columns)

input_matches = [match for matches in hybrid_df["Lexical_Matches"] for match in matches]
output_matches = [match for matches in semantic_enrichment_df["Lexical_Matches"] for match in matches]

root_mismatches = sum(a.get("Root_AR") != b.get("Root_AR") for a, b in zip(input_matches, output_matches))
source_mismatches = sum(a.get("Root_Source") != b.get("Root_Source") for a, b in zip(input_matches, output_matches))
invented_unresolved_roots = sum(
    (not bool(a.get("Root_Resolved", False)))
    and (b.get("Root_AR") is not None and str(b.get("Root_AR")).strip() != "")
    for a, b in zip(input_matches, output_matches)
)

if root_mismatches or source_mismatches or invented_unresolved_roots:
    raise AssertionError("Authoritative Point-2 root lineage changed during semantic enrichment")

term_semantic_records = []
for match in output_matches:
    term_semantic_records.append({
        "Root_Resolved": bool(match.get("Root_Resolved", False)),
        "Root_Source": match.get("Root_Source"),
        "Lookup_Attempted": bool(match.get("Maqayis_Lookup_Attempted", False)),
        "Semantic_Found": bool(match.get("Semantic_Found", False)),
    })
term_semantic_df = pd.DataFrame(term_semantic_records)

print("=" * 70)
print("SEMANTIC ENRICHMENT COMPLETED")
print("=" * 70)
print(f"Questions Processed        : {len(semantic_enrichment_df):,}")
print(f"Query Terms                : {len(term_semantic_df):,}")
print(f"Maqāyīs Lookup Attempts    : {int(term_semantic_df['Lookup_Attempted'].sum()):,}")
print(f"Successful Semantic Matches: {int(term_semantic_df['Semantic_Found'].sum()):,}")
print(f"Root Mismatches            : {root_mismatches:,}")
print(f"Root-Source Mismatches     : {source_mismatches:,}")
print(f"Invented Unresolved Roots  : {invented_unresolved_roots:,}")

coverage_by_source = (
    term_semantic_df.loc[term_semantic_df["Root_Resolved"]]
    .groupby("Root_Source", dropna=False)["Semantic_Found"]
    .agg(Terms="size", Semantic_Matches="sum")
    .reset_index()
)
coverage_by_source["Coverage_Percent"] = 100 * coverage_by_source["Semantic_Matches"] / coverage_by_source["Terms"]
print("\nSemantic Coverage by Authoritative Root Source")
display(coverage_by_source)

SEMANTIC ENRICHMENT COMPLETED
Questions Processed        : 251
Query Terms                : 1,342
Maqāyīs Lookup Attempts    : 1,013
Successful Semantic Matches: 724
Root Mismatches            : 0
Root-Source Mismatches     : 0
Invented Unresolved Roots  : 0

Semantic Coverage by Authoritative Root Source


,Root_Source,Terms,Semantic_Matches,Coverage_Percent
0,CAMeL,488,351,71.926230
1,QAC,525,373,71.047619


## Exporting the Corrected Semantic-Enrichment Dataset

The complete semantically enriched query representation is exported to an isolated corrected-output location. `Query_Terms`, `Query_Term_Provenance`, and `Lexical_Matches` are serialized as standards-compliant JSON.

The export preserves the final Point-2 query roots and provenance and adds explicit Maqāyīs lookup-attempt, lookup-root, match-status, evidence, and source fields. It is the corrected query-side input intended for the later controlled RALSR lineage stage.

In [13]:
# ============================================================
# Export Corrected Semantic Enrichment Dataset
# ============================================================

export_df = semantic_enrichment_df.copy()

for column in NESTED_COLUMNS:
    export_df[column] = export_df[column].apply(
        lambda value: json.dumps(value, ensure_ascii=False, allow_nan=False)
    )

export_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

reloaded_df = pd.read_csv(OUTPUT_FILE)
if len(reloaded_df) != len(export_df):
    raise AssertionError("CSV reload changed the number of question records")
if reloaded_df["Question_ID"].tolist() != export_df["Question_ID"].tolist():
    raise AssertionError("CSV reload changed Question_ID ordering")
for column in NESTED_COLUMNS:
    reloaded_df[column].map(json.loads)

print("=" * 70)
print("CORRECTED SEMANTIC ENRICHMENT DATASET EXPORTED")
print("=" * 70)
print(f"Questions Exported : {len(export_df):,}")
print(f"Output File        : {OUTPUT_FILE.name}")
print(f"Location           : {OUTPUT_DIR}")
print("CSV reload and strict JSON parsing completed successfully.")

CORRECTED SEMANTIC ENRICHMENT DATASET EXPORTED
Questions Exported : 251
Output File        : QuranQA_Semantic_Enrichment_Corrected.csv
Location           : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Maqāyīs Lookup\Corrected_Outputs\Notebook_7_Maqayis_Semantic_Enrichment
CSV reload and strict JSON parsing completed successfully.


## Notebook Summary

This corrected notebook attached *Maqāyīs al-Lughah* lexical-semantic evidence to the final authoritative Point-2 query roots without rerunning root extraction or changing any root decision. It processed 1,342 query-term records: 1,013 strict valid-root terms (525 QAC and 488 CAMeL) and 329 unresolved terms.

Exact lookup found Maqāyīs evidence for 724 resolved term occurrences and no entry for 289. This is 71.47% coverage over strict valid-root terms and 53.95% over all query terms. QAC-derived terms matched 373 of 525 (71.05%); CAMeL-derived terms matched 351 of 488 (71.93%). The 393 unique authoritative roots produced 295 exact lexicon matches and 98 no-match roots.

All Point-2 roots, sources, statuses, and detailed QAC/CAMeL provenance were preserved. Root substitutions, source substitutions, and invented roots for unresolved terms were all zero. Maqāyīs evidence remains an exact-root lexical resource attached to the authoritative query representation; it is not treated as a root generator or evidence of passage relevance.